# D2-05 Regionalised methods across impact categories

This notebook compares two impact categories where regionalization matters a lot: water scarcity and biodiversity. We stay with a toy system so that the focus remains on interpretation.


## Learning goals

After this notebook, you should be able to:

- explain why different regionalized methods respond differently to the same inventory
- compare category-specific contribution patterns on one shared product system
- distinguish a category-by-category interpretation from a compact cross-category summary
- identify when a change in ranking is caused by the characterization factors rather than by the inventory itself


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023
- Scherer, L., Rosa, F., Sun, Z., Michelsen, O., De Laurentiis, V., Marques, A., Pfister, S., Verones, F., & Kuipers, K. J. J. (2023). Biodiversity Impact Assessment Considering Land Use Intensities and Fragmentation. *Environmental Science & Technology, 57*(48), 19612-19623. https://doi.org/10.1021/acs.est.3c04191


## 1) Reuse the same small inventory matrix

To isolate the effect of the impact category, we keep the same compact CAM inventory from `D2-02`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

activities = [
    'Li+ extraction (CL)',
    'Hydropower (CL)',
    'Li2CO3 production (CN)',
    'Hydropower (CN)',
    'CAM production (DE)',
    'Hydropower (DE)',
]
flows = ['Water, underground [kg]', 'Water, surface [kg]', 'Water, unspecified [kg]']

A = np.array([
    [ 1.0,  0.0, -0.8, 0.0, -0.1, 0.0],
    [-0.5,  1.0,  0.0, 0.0,-25.0, 0.0],
    [ 0.0,  0.0,  1.0, 0.0, -0.2, 0.0],
    [ 0.0,  0.0, -3.0, 1.0,  0.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,  1.0, 0.0],
    [ 0.0,  0.0,  0.0, 0.0,-25.0, 1.0],
])
B = np.array([
    [2.0, 0.0, 0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 3.0, 0.0, 0.0, 0.0],
    [0.0, 0.02, 0.0, 0.015, 0.2, 0.005],
])
f = np.array([0.0, 0.0, 0.0, 0.0, 1.0, 0.0])

s = np.linalg.solve(A, f)
G = pd.DataFrame(B @ np.diag(s), index=flows, columns=activities)
G


## 2) Build two toy regionalized method matrices

These are **teaching coefficients**, not official AWARE2.0 or biodiversity CFs. Their purpose is to make the category differences visible on a small example.


In [ ]:
C_aware = pd.DataFrame(0.0, index=flows, columns=activities)
C_aware.loc['Water, underground [kg]', 'Li+ extraction (CL)'] = 50.0
C_aware.loc['Water, surface [kg]', 'Li2CO3 production (CN)'] = 35.0
C_aware.loc['Water, unspecified [kg]', 'Hydropower (CL)'] = 8.0
C_aware.loc['Water, unspecified [kg]', 'Hydropower (CN)'] = 6.0
C_aware.loc['Water, unspecified [kg]', 'CAM production (DE)'] = 12.0
C_aware.loc['Water, unspecified [kg]', 'Hydropower (DE)'] = 5.0

C_biodiversity = pd.DataFrame(0.0, index=flows, columns=activities)
C_biodiversity.loc['Water, underground [kg]', 'Li+ extraction (CL)'] = 18.0
C_biodiversity.loc['Water, surface [kg]', 'Li2CO3 production (CN)'] = 55.0
C_biodiversity.loc['Water, unspecified [kg]', 'Hydropower (CL)'] = 4.0
C_biodiversity.loc['Water, unspecified [kg]', 'Hydropower (CN)'] = 10.0
C_biodiversity.loc['Water, unspecified [kg]', 'CAM production (DE)'] = 7.0
C_biodiversity.loc['Water, unspecified [kg]', 'Hydropower (DE)'] = 3.0

display(C_aware)
display(C_biodiversity)


## 3) Characterize the same inventory in two different categories

Because `G` stays the same, any change in interpretation comes from the impact category and its regionalized factors.


In [ ]:
H_aware = C_aware * G
H_biodiversity = C_biodiversity * G

category_totals = pd.DataFrame({
    'AWARE-like total': H_aware.sum(axis=0),
    'Biodiversity-like total': H_biodiversity.sum(axis=0),
})
category_totals


## 4) Category-by-category interpretation

First, inspect the characterized inventories directly.


In [ ]:
print('AWARE-like characterized inventory')
display(H_aware)

print('Biodiversity-like characterized inventory')
display(H_biodiversity)


## Checkpoint 1

Which activity is the main contributor in the AWARE-like case, and which one is the main contributor in the biodiversity-like case?


In [ ]:
# TODO
# aware_top = ...
# biodiversity_top = ...


In [ ]:
aware_top = H_aware.sum(axis=0).sort_values(ascending=False).index[0]
biodiversity_top = H_biodiversity.sum(axis=0).sort_values(ascending=False).index[0]

print('AWARE-like dominant activity:', aware_top)
print('Biodiversity-like dominant activity:', biodiversity_top)


## 5) Compact comparative overview

Once you understand each category separately, a compact summary is useful to compare rankings.


In [ ]:
summary = pd.DataFrame([
    {
        'category': 'AWARE-like water scarcity',
        'total score': float(H_aware.to_numpy().sum()),
        'top activity': H_aware.sum(axis=0).sort_values(ascending=False).index[0],
        'top flow': H_aware.sum(axis=1).sort_values(ascending=False).index[0],
    },
    {
        'category': 'Biodiversity-like',
        'total score': float(H_biodiversity.to_numpy().sum()),
        'top activity': H_biodiversity.sum(axis=0).sort_values(ascending=False).index[0],
        'top flow': H_biodiversity.sum(axis=1).sort_values(ascending=False).index[0],
    },
])
summary


In [ ]:
plot_data = category_totals.rename_axis('activity').reset_index().melt(
    id_vars='activity',
    var_name='category',
    value_name='score'
)

fig, ax = plt.subplots(figsize=(10, 4.5))
for category, group in plot_data.groupby('category'):
    ax.bar(group['activity'] + ' | ' + category, group['score'], label=category)
ax.set_ylabel('Characterized contribution')
ax.set_title('Same inventory, different regionalized method logic')
plt.xticks(rotation=80)
plt.tight_layout()
plt.show()


## Recap

After this notebook, you should now be able to:

- explain why the same inventory can rank differently across regionalized categories
- separate inventory effects from method effects
- interpret water-scarcity and biodiversity results category by category
- summarize category differences in a compact comparison table or plot
